# RF-Based Drone Detection and Classification Using Deep Learning
### UC Berkeley ML/AI Professional Certificate — Capstone Project
**Author:** Aya Mundhenk  
**GitHub:** https://github.com/ayamundhenk/berkeley_capstone


## 1. Problem Statement

**Research Question:**  
Can radio frequency (RF) signal data alone — without cameras, radar, or GPS — be used to reliably detect the presence of a drone and classify which type it is?

### Background

The rapid growth of consumer and commercial unmanned aerial vehicles (UAVs) has created growing security and airspace management challenges, particularly in restricted or sensitive environments such as airports, prisons, stadiums, and military installations. Most existing counter-drone systems rely on cameras (fail at night/poor weather), radar (struggles with small/slow drones), or GPS tracking (only works if the drone cooperates). RF-based detection closes that gap: **every drone must communicate with its controller**, whether or not it wants to be tracked.

### Goals

This project builds a machine learning model that:
1. **Detects** whether a drone is present in a given RF environment (binary classification)
2. **Classifies** the specific drone type — AR, Bebop, or Phantom — and its operating mode (multi-class classification)

Performance will be evaluated across varying signal-to-noise (SNR) conditions to assess real-world reliability alongside Bluetooth and Wi-Fi interference in the shared 2.4 GHz band.

## 2. Data Sources & Structure


In [2]:
import os, re, sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal as scipy_signal, io as sio
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "| Pandas:", pd.__version__)

Python: 3.13.9
NumPy: 2.3.5 | Pandas: 2.3.3


### 2.1 DroneRF Dataset

**Source:** Allahham et al. (2019), Mendeley Data  
**Link:** https://data.mendeley.com/datasets/f4c2b4n755/1  
**Size:** ~40 GB | **Format:** CSV (one row per file = 1,000,000 amplitude samples)

The dataset contains **227 recorded segments** from three drones — Parrot Bebop, Parrot AR, and DJI Phantom — each operating in four flight modes, plus background RF recordings with no drone present. Each segment is split into two files capturing the low and high halves of the 2.4 GHz band.

**Flight modes recorded:**
| Mode | Description |
|---|---|
| Mode 1 | On and connected to controller |
| Mode 2 | Hovering (no manual input) |
| Mode 3 | Flying without video recording |
| Mode 4 | Flying with video recording |

**Filename label scheme (BUI — Binary Unique Identifier):**

Filenames encode all label information, e.g. `11010H3.csv`:

| BUI bits | Meaning |
|---|---|
| Bit 1 | `0` = background (no drone), `1` = drone present |
| Bits 2–3 | Drone type: `00`=AR, `01`=Bebop, `10`=Phantom |
| Bits 4–5 | Flight mode: `00`=Mode1, `01`=Mode2, `10`=Mode3, `11`=Mode4 |
| `L` or `H` | Low or High frequency half of the spectrum |
| Trailing number | Segment index |


## 3. Data Loading & Structure Exploration

In [4]:
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ── PATH SETUP ─────────────────────────────────────────────────────────────────
# The DroneRF dataset is in this shared Google Drive folder:
#   https://drive.google.com/drive/folders/1vdDid4Yv80pnLa7zDTY2twca3YooGxxG

DRONERF_ROOT = "/content/drive/MyDrive/DroneRF" if IN_COLAB else "./data/DroneRF"

CONFIG = {
    "dronerf_root": DRONERF_ROOT,
    "processed_dir": "/content/drive/MyDrive/capstone_processed" if IN_COLAB else "./data/processed",
}

os.makedirs(CONFIG["processed_dir"], exist_ok=True)
print("DroneRF root:", CONFIG["dronerf_root"])
print("Processed output:", CONFIG["processed_dir"])


DroneRF root: ./data/DroneRF
Processed output: ./data/processed
